# T1 - Deep Learning 

Feito por: Louise e Luca


Numpy faz as operacoes de matrizes para nos, então nos temos que criar uma classe base (pytorch chama de tensor) 

tensor --> matriz pessos + matriz de gradiente 

In [12]:
import numpy as np

Unidade basica - matriz que guarda as informcoes para a fazer o back propagation depois 

In [13]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.random.random(dimencao)

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        return self.peso + outra.peso

In [14]:
dimencao = [2,3]
exemplo = Tensor(dimencao)

print(exemplo)
print(exemplo.gradiente)

[[0.45263783 0.52530103 0.42468652]
 [0.29926874 0.05547745 0.3284156 ]]
[[0.72090218 0.09241017 0.63221797]
 [0.42846164 0.05827481 0.60344236]]


In [15]:
exemplo2 = Tensor(dimencao)

In [16]:
exemplo + exemplo2

array([[1.33214576, 0.73836816, 0.97262142],
       [0.93983752, 0.58453973, 0.79618196]])

Agora vamos mexer no gradiente 
queremos impemetar as funcoes basicas para qaundo fizermos o back propagetion qualquer funcao ser diferenciavel (derivar )

uma forma de ccalular o gradiente: graudar as regras da derivada 


In [17]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.random.random(dimencao)
        self.regra = 0

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        self.regra = 1 
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso + outra.peso

        return retorno

    def __mul__(self, outra):  # ponto a ponto 

        self.regra = outra.peso
        outra.regra = self.peso


        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso * outra.peso
 
        return retorno
    

    

In [18]:
dimencao = [2,2]
teste1 = np.array([[1,2],[3,4]])
teste2 = np.array([[1,1],[1,1]])

exemplo_3 = Tensor(dimencao)
exemplo_4 = Tensor(dimencao)

exemplo_3.peso = teste1
exemplo_4.peso = teste2

print(exemplo_3, "\n\n", exemplo_4)

[[1 2]
 [3 4]] 

 [[1 1]
 [1 1]]


In [19]:
exemplo_4 * exemplo_3

In [20]:
exemplo_3.regra

array([[1, 1],
       [1, 1]])

In [21]:
exemplo_4.regra

array([[1, 2],
       [3, 4]])

Agora temos o nosso gradiente local, nos chamamos isso de regra, aqui esta uma teste para ver se esta funcionando:

dervada em relacao a 3 da funcao: ( (3 * 4) * 5 ) = 20 

In [22]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

print(exemplo_num3, "\n", exemplo_num4, "\n", exemplo_num5)
print( exemplo_num3 * exemplo_num4 * exemplo_num5)

print(exemplo_num5.regra) # 3 * 4 = 12 o gradiente esta certo
exemplo_num5.gradiente = exemplo_num5.regra 

print(exemplo_num5.gradiente)

print(exemplo_num3.regra)
exemplo_num3.gradiente = exemplo_num3.regra * exemplo_num5.peso

print(exemplo_num3.gradiente)

[3] 
 [4] 
 [5]
[60]
[12]
[12]
[4]
[20]


In [23]:
exemplo_3.peso.shape

(2, 2)

Agora nos queremos que atualize o gradiente automaticamente, sem ter que fazer 
```python 
exemplo_num3.gradiente = exemplo_num3.regra * exemplo_num5.peso
```

para isso a gente vai mudar para fazwe uma funcao= gradiente local * outro

In [24]:
class Tensor:
    def __init__(self, dimencao):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda x:0
        self.prox = None

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso + outra.peso
        self.prox = retorno # agora temos um grafo

        def regra(tensor):
            tensor.gradiente += self.prox.gradiente

        self.regra = lambda: regra(self)
        outra.regra = lambda: regra(outra)

        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        retorno = Tensor(self.peso.shape)
        retorno.peso = self.peso * outra.peso
        self.prox = retorno

        def regra():
            self.gradiente += self.prox.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += outra.prox.gradiente * self.peso 
 
        return retorno

    def backward(self): # calulo dos gradientes
        pass



    

In [25]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + exemplo_num4
teste.gradiente = np.ones_like(teste, float) 
exemplo_num3.regra()
print(exemplo_num3.gradiente)

[1.]


Vamos trabalhar numa forma coesa de fazer o backward pass automaticmaente, depois voltamos a implementar funções. Usaremos o algoritmo de ordenação topologica baseado em DFS. Para isso vamos mudar a lógica de prox, e definir um anterior, pois é consideravelmente mais facil trabalhar na ordem inversa

In [26]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        retorno = Tensor(self.peso.shape, (self, outra))
        retorno.peso = self.peso + outra.peso
        self.prox = retorno # agora temos um grafo

        def regra():
            self.gradiente += retorno.gradiente
            outra.gradiente += retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        retorno = Tensor(self.peso.shape, (self, outra))
        retorno.peso = self.peso * outra.peso
        self.prox = retorno

        def regra():
            self.gradiente += retorno.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += retorno.gradiente * self.peso 
        
        retorno.regra = regra
 
        return retorno

    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [27]:
dimencao = [1]
num3 = np.array([3])
num4 = np.array([4])
num5 = np.array([5])

exemplo_num3 = Tensor(dimencao)
exemplo_num4 = Tensor(dimencao)
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + exemplo_num4
final = exemplo_num5 * teste
final.backward()
print(exemplo_num3.gradiente)

[5.]


Agora vamos completar as 4 operações basicas e permitir operações com escalares

In [28]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # AS duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = outra
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente
            outra.gradiente += retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = outra
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += retorno.gradiente * self.peso 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        return self.__mul__( other**-1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
        

    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [29]:
dimencao = [1]
num3 = np.array([3,4,5])
num4 = np.array([4,8,9])
num5 = np.array([5])

exemplo_num3 = Tensor([3])
exemplo_num4 = Tensor([3])
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 * 10
print(exemplo_num3)

[3. 4. 5.]


Agora temos as operações básicas, mas isso quebra o nosso sistema de backward pass, por causa de erros de dimensão, então precisamos adaptar as dimensõoes

In [30]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # As duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)
        
        
    # Função responsavel por permitir o backward pass mesmo sob broadcast
    def soma_dimensional(self, gradiente, shape):
        while gradiente.ndim > len(shape): # Isso elimina dimensões extras adicionadas via boradcasting pelo numpy
            gradiente = gradiente.sum(axis=0)
            
        for i, dim in enumerate(shape): # Isso reduz a dimensionalidade de dimensões existentes de volta para 1 (caso originalmente fosse um)
            if dim == 1 and gradiente.shape[i] != 1:
                gradiente = gradiente.sum(axis=i, keepdims=True) # keepdims = True mantem a dimensão original do tensor, na pratica ele adiciona um parenteses e mantem como dimensão de tamanho 1
                
        return gradiente
            
        

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente, self.peso.shape)
            outra.gradiente += self.soma_dimensional(retorno.gradiente, outra.peso.shape)
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente * outra.peso # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += retorno.gradiente * self.peso 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        return self.__mul__( other**-1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
        

    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [31]:
a = np.array([[[1,2,3],[4,5,6]], [[7,8,9], [20,12,11]]])

a.sum(axis=1, keepdims=True).shape

(2, 1, 3)

In [32]:
dimencao = [1]
num3 = np.array([3,4,5])
num4 = np.array([4,8,9])
num5 = np.array([5])

exemplo_num3 = Tensor([3])
exemplo_num4 = Tensor([3])
exemplo_num5 = Tensor(dimencao)

exemplo_num3.peso = num3
exemplo_num4.peso = num4
exemplo_num5.peso = num5

teste = exemplo_num3 + 10
teste.backward()
print(exemplo_num5.gradiente)

[0.]


In [33]:
exemplo_num3 = Tensor([3])
exemplo_num3.peso = np.array([3,4,5])

a = exemplo_num3 ** (0.5)
print(a)

[1.73205081 2.         2.23606798]


Agora,temos que implementar as últimas funções que ainda faltam, que são produto matricial, log, radicialização e somatorio

In [34]:
class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.random.random(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # As duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)
        
        
    # Função responsavel por permitir o backward pass mesmo sob broadcast
    def soma_dimensional(self, gradiente, shape):
        while gradiente.ndim > len(shape): # Isso elimina dimensões extras adicionadas via boradcasting pelo numpy
            gradiente = gradiente.sum(axis=0)
            
        for i, dim in enumerate(shape): # Isso reduz a dimensionalidade de dimensões existentes de volta para 1 (caso originalmente fosse um)
            if dim == 1 and gradiente.shape[i] != 1:
                gradiente = gradiente.sum(axis=i, keepdims=True) # keepdims = True mantem a dimensão original do tensor, na pratica ele adiciona um parenteses e mantem como dimensão de tamanho 1
                
        return gradiente
            
        

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente, self.peso.shape)
            outra.gradiente += self.soma_dimensional(retorno.gradiente, outra.peso.shape)
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente * outra.peso, self.peso.shape) # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += self.soma_dimensional(retorno.gradiente * self.peso, outra.peso.shape) 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        return self.__mul__( other**-1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
    
    def root(self, value = 2): # Podemos aproveitar a implementação em potencia para fazer a radicialização
        return self ** (1/value)
    
    def __matmul__(self, outra):
        a = self.peso @ outra.peso
        retorno = Tensor(a.shape, (self, outra))
        retorno.peso = a
        
        def regra():
            self.gradiente +=retorno.gradiente @ outra.peso.T
            outra.gradiente += self.peso.T @ retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    
    def log(self, base="euler"):
        if base == "euler":
            resultado = np.log(self.peso)
            retorno = Tensor(resultado.shape, (self,))
            retorno.peso = resultado
            
            def regra():
                self.gradiente += retorno.gradiente/self.peso
                
            retorno.regra = regra
            
            return retorno
        
        resultado = np.log(self.peso) / np.log(base) # Propriedade de mudança de base dos logaritmos
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente / (self.peso * np.log(base))

        retorno.regra = regra

        return retorno
    
    def sum(self, axis=0):
        resultado = np.sum(self.peso, axis=axis)
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            
            gradiente = retorno.gradiente
            
            if axis is not None: # Isso é para impedir o bug de caso o resultado final do sum no foarward tenha sido um escalar
                # Expandindo a dimensão removida durante o foarward
                gradiente = np.expand_dims(gradiente, axis=axis)
            
            self.gradiente += np.broadcast_to(gradiente, self.peso.shape) # Copia as dimensões em 1 para bater com o valor da matriz
            
            
        retorno.regra = regra
        
        return retorno
        

        
        
            
    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()


In [35]:
exemplo_num3 = Tensor([3,2])
exemplo_num4 = Tensor([2,3])

exemplo_num3.peso = np.array([[1,2],
                              [3,4], 
                              [5,6]])
exemplo_num4.peso = np.array([[1,2,3], [4,5,6]])

teste = exemplo_num3 @ exemplo_num4
teste2 = teste + teste
print(teste)
print(teste2)



[[ 9. 12. 15.]
 [19. 26. 33.]
 [29. 40. 51.]]
[[ 18.  24.  30.]
 [ 38.  52.  66.]
 [ 58.  80. 102.]]


In [36]:
exemplo_num3 = Tensor([3,2])
exemplo_num4 = Tensor([2,3])

exemplo_num3.peso = np.array([[1,2],
                              [3,4], 
                              [5,6]])
exemplo_num4.peso = np.array([[1,2,3], [4,5,6]])

teste = exemplo_num3.log(2).sum()
teste.backward()
print(teste)
print(teste.gradiente)



[3.9068906 5.5849625]
[1. 1.]


Abaixo seguem apenas testes e detlahes da adaptação do autograd para ele ter as ferramentas para facilitar a criação de mlps

In [37]:
import numpy as np

class Tensor:
    def __init__(self, dimencao, filhos= ()):
        self.peso = np.zeros(dimencao)
        self.gradiente = np.zeros(dimencao,float)
        self.regra = lambda: None # Mudança ja que agora regra não irá receber um valor
        self.anteriores = set(filhos)  # Adicionamos isso para poder calcular a ordem topologic
    
    # As duas property abaixo servem para garantir que peso sempre seja um float, isso é necessario para evitar o problema de que o numpy não permite que um inteiro seja elevado a uma potência negativa
    @property
    def peso(self):
        return self._peso

    @peso.setter
    def peso(self, valor):
        self._peso = np.asarray(valor, dtype=float)
        
        
    # Função responsavel por permitir o backward pass mesmo sob broadcast
    @staticmethod
    def soma_dimensional(gradiente, shape):
        while gradiente.ndim > len(shape): # Isso elimina dimensões extras adicionadas via boradcasting pelo numpy
            gradiente = gradiente.sum(axis=0)
            
        for i, dim in enumerate(shape): # Isso reduz a dimensionalidade de dimensões existentes de volta para 1 (caso originalmente fosse um)
            if dim == 1 and gradiente.shape[i] != 1:
                gradiente = gradiente.sum(axis=i, keepdims=True) # keepdims = True mantem a dimensão original do tensor, na pratica ele adiciona um parenteses e mantem como dimensão de tamanho 1
                
        return gradiente
            
        

    def __str__(self):
        return str(self.peso) # se for chamada vai retornar so o peso 

    def __add__(self, outra):
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
        
        resultado = self.peso + outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente, self.peso.shape)
            outra.gradiente += self.soma_dimensional(retorno.gradiente, outra.peso.shape)
            
        retorno.regra = regra
        
        return retorno
    

    def __mul__(self, outra):  # ponto a ponto 
        if isinstance(outra, (int, float)):
            a = Tensor([1])
            a.peso = np.array([outra])
            outra = a
            
        
        resultado = self.peso * outra.peso
        retorno = Tensor(resultado.shape, (self, outra))
        retorno.peso = resultado

        def regra():
            self.gradiente += self.soma_dimensional(retorno.gradiente * outra.peso, self.peso.shape) # += pq soma os dois gradientes p um tensor com 2 saidades vai alterar x + y no final (por isso somamos)
            outra.gradiente += self.soma_dimensional(retorno.gradiente * self.peso, outra.peso.shape) 
        
        retorno.regra = regra
 
        return retorno
    
    def __pow__(self, outra): # Exponenciação (outra aqui é um escalar). Usaremos isso para fazer a divisão apartir da multiplicação
            retorno  = Tensor(self.peso.shape, (self,))
            retorno.peso = self.peso ** outra
            
            def regra():
                self.gradiente += outra * (self.peso ** (outra-1)) * retorno.gradiente  
                
            retorno.regra = regra
            
            return retorno   
    
    def __rmul__(self, outra): # Permitir multiplicação por escalar com o escalar do lado direito
        return self.__mul__(outra)
    
    def __radd__(self, outra): # Permitir soma por escalar com o escalar do lado direito
        return self.__add__(outra)
    
    def __neg__(self):
        return self * -1
    
    def __sub__(self, outra):
        return self.__add__(-outra)
    
    def __rsub__(self, outra):
        return outra + (-self)   
        
    def __truediv__(self, other):
        if isinstance(other, (int, float, np.number)):
            return self * (1 / other)
        return self * (other ** -1)
    
    def __rtruediv__(self, other):
        return (self **-1) * other
    
    def __matmul__(self, outra):
        a = self.peso @ outra.peso
        retorno = Tensor(a.shape, (self, outra))
        retorno.peso = a
        
        def regra():
            self.gradiente +=retorno.gradiente @ outra.peso.T
            outra.gradiente += self.peso.T @ retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    
    def log(self, base="euler"):
        if base == "euler":
            resultado = np.log(self.peso)
            retorno = Tensor(resultado.shape, (self,))
            retorno.peso = resultado
            
            def regra():
                self.gradiente += retorno.gradiente/self.peso
                
            retorno.regra = regra
            
            return retorno
        
        resultado = np.log(self.peso) / np.log(base) # Propriedade de mudança de base dos logaritmos
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente / (self.peso * np.log(base))

        retorno.regra = regra

        return retorno

    def root(self, value = 2): # Podemos aproveitar a implementação em potencia para fazer a radicialização
            return self ** (1/value)
    
    def sum(self, axis=None):
        resultado = np.sum(self.peso, axis=axis)
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            
            gradiente = retorno.gradiente
            
            if axis is not None: # Isso é para impedir o bug de caso o resultado final do sum no foarward tenha sido um escalar
                # Expandindo a dimensão removida durante o foarward
                gradiente = np.expand_dims(gradiente, axis=axis)
            
            self.gradiente += np.broadcast_to(gradiente, self.peso.shape) # Copia as dimensões em 1 para bater com o valor da matriz
            
            
        retorno.regra = regra
        
        return retorno
    
    
    
    # Funções de ativação

    def reLU(self):
        retorno = Tensor(self.peso.shape, (self,))
        retorno.peso = np.where(self.peso>0, self.peso, 0)

        def regra():
            self.gradiente += (self.peso > 0) * retorno.gradiente

        retorno.regra = regra

        return retorno
    
    
    def sigmoid(self):
        resultado = 1 / (1 + np.exp(-self.peso))

        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado

        def regra():
            self.gradiente += retorno.gradiente * retorno.peso * (1 - retorno.peso)

        retorno.regra = regra

        return retorno
    
    
    def tanh(self):
        resultado = np.tanh(self.peso)
        retorno = Tensor(resultado.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            self.gradiente += (1 - resultado**2) * retorno.gradiente
            
        retorno.regra = regra
        
        return retorno
    
    def linear(self): # Essa aqui foi difícil 
        return self
    
    def leakyReLU(self, alpha=0.01):
        resultado = np.where(self.peso>0, self.peso, alpha*self.peso)
        retorno = Tensor(self.peso.shape, (self,))
        retorno.peso = resultado
        
        def regra():
            self.gradiente += np.where(self.peso>0, 1, alpha) * retorno.gradiente
            
        retorno.regra = regra
        
        return retorno

            
    def backward(self): # calulo dos gradientes
        self.gradiente = np.ones_like(self.peso, dtype=float)
        
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.anteriores:
                    build_topo(child)
                topo.append(v) # Note que quando chamamos a função com self, ele será o ultimo a ser visitado ja que é um dfs
        build_topo(self)
        
        for nodo in reversed(topo):
            nodo.regra()
            
    def __call__(self, outro):
        return self @ outro



def Dense(input, output, inicializacao = "He", limite_inferior=0, limite_superior=1, desvio_padrao=1, media=0):
    matriz = Tensor([input, output])
    
    if inicializacao == "He":
        matriz.peso = np.random.normal(0,
                                       np.sqrt( 2/input),
                                       size=(input,output))
    elif inicializacao == "Xavier":
        matriz.peso = np.random.normal(0,
                                       np.sqrt( 2/(input+output)),
                                       size=(input,output))
                                       
    elif inicializacao == "Uniforme":
        matriz.peso = np.random.uniform(limite_inferior, limite_superior, size=(input, output))
    elif inicializacao == "Normal":
        matriz.peso = np.random.normal(media,
                                       np.sqrt(desvio_padrao), 
                                       size=(input, output))
        
    return matriz


class MLP:
    def __init__(self, *camadas):
        self.camadas = camadas

    def __call__(self, x):
        for pesos, ativacao in self.camadas:
            x = x @ pesos
            x = getattr(x, ativacao)()

        return x





def MSE(y, y_pred):
    custo = ((y - y_pred)**2).sum()
    return custo / y.peso.size

def BCE(y, y_pred):
    custo = -(y * y_pred.log() + (1-y) * (1-y_pred).log())
    return custo.sum() / y.peso.size



In [38]:
exemplo_num3 = Tensor([3,2])
exemplo_num4 = Tensor([2,3])

exemplo_num3.peso = np.array([[1,2],
                              [3,4], 
                              [5,6]])
exemplo_num4.peso = np.array([[1,2,3], [4,5,6]])

teste = exemplo_num3.sigmoid()
teste.backward()
print(teste)
print(teste.gradiente)



[[0.73105858 0.88079708]
 [0.95257413 0.98201379]
 [0.99330715 0.99752738]]
[[1. 1.]
 [1. 1.]
 [1. 1.]]


In [42]:
modelo = MLP(
    (Dense(3, 5, "He"), "reLU"),
    (Dense(5, 4, "He"), "reLU"),
    (Dense(4, 1, "Xavier"), "sigmoid")
)

x = Tensor((10, 3))
x.peso = np.random.randn(10, 3)
y = Tensor((10, 1))
y.peso = np.random.randint(0, 2, (10, 1))

y_pred = modelo(x)

print(modelo(x)), print( y)

[[0.38400366]
 [0.19875949]
 [0.32144723]
 [0.20608467]
 [0.44637847]
 [0.17525102]
 [0.38253926]
 [0.38695151]
 [0.21545328]
 [0.39383096]]
[[0.]
 [0.]
 [0.]
 [1.]
 [0.]
 [0.]
 [0.]
 [1.]
 [1.]
 [0.]]


(None, None)